# Day 46: Implement "Persistence" in LangGraph

Welcome to Day 46! Today we are focusing on **Persistence** in LangGraph. By default, LangGraph states are ephemeral—they only exist for the duration of a single run. To build useful agents (like conversational bots or long-running tasks), we need the agent to "remember" previous interactions.

## Core Theory (Just-in-Time)

### The "Why"
When a user chats with an agent, they expect the agent to remember context from five minutes ago. In a stateless system, the agent forgets everything as soon as the graph execution finishes. Persistence solves this by saving the graph's State at every step to a storage layer (like SQLite or Postgres) and reloading it when the user returns.

### The "How"
LangGraph uses a concept called **Checkpointers**. A checkpointer acts as a middleman between the graph execution and the database.
1.  **State Save:** After every node execution, the checkpointer saves the current state snapshot (including messages, variables, etc.).
2.  **Thread ID:** Each conversation is assigned a unique `thread_id`. 
3.  **State Load:** When starting a new execution, you provide the `thread_id` in a `RunnableConfig` object. LangGraph automatically fetches the last state from the checkpointer and resumes execution from there.

We'll use LangGraph's built-in `MemorySaver` (an in-memory checkpointer) for this lesson, but the concept is identical when using a persistent database checkpointer (like `AsyncSqliteSaver` or `AsyncPostgresSaver`).

## Common Pitfalls in Production
1.  **State Size Bloat:** Storing every single message in a long conversation will eventually exceed the LLM's context window and slow down database reads/writes. In production, you must implement state pruning or summarization nodes to keep the state manageable.
2.  **Thread Collision:** Accidentally hardcoding or reusing the same `thread_id` for different users means User A will suddenly see User B's conversation history. Always generate unique thread IDs (e.g., UUIDs).
3.  **Unserializable State Types:** The checkpointer needs to save the state to a database (often as JSON or pickle). If your State object contains unserializable items (like a raw database connection object or a file handle), the checkpointer will crash.

## Setup and Dependencies

First, let's ensure we have the necessary packages. We need `langgraph`, `langchain-openai`, and `langchain-core`.

In [1]:
# Run this in your terminal if you haven't already:
# uv pip install langgraph langchain-openai langchain-core

## 1. Implementing a Persistent Graph

Let's build a simple chatbot that remembers our name using `MemorySaver`. We will define a basic graph with a single node that calls an LLM.

In [2]:
import os
from typing import Annotated, TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

# Ensure your API key is set
# os.environ["OPENAI_API_KEY"] = "your-api-key"

# 1. Define the State
# We use `add_messages` to append new messages instead of overwriting the list
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

# 2. Define the LLM Node
def chatbot_node(state: AgentState):
    """Calls the LLM and appends the response to the messages list."""
    # In a real environment, initialize the LLM outside the node if possible
    # to avoid re-instantiation, but this is fine for demonstration.
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
    
    # We wrap the invocation in a try-except to handle local testing 
    # environments where OPENAI_API_KEY might be a dummy value.
    try:
        response = llm.invoke(state["messages"])
        return {"messages": [response]}
    except Exception as e:
        print(f"[Mock] LLM API Call Failed: {e}")
        from langchain_core.messages import AIMessage
        return {"messages": [AIMessage(content="I am a mock response because the API key is invalid.")]}


# 3. Build the Graph
builder = StateGraph(AgentState)
builder.add_node("chatbot", chatbot_node)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

# 4. Initialize the Checkpointer
# MemorySaver stores state in memory. For production, use SqliteSaver or PostgresSaver.
memory = MemorySaver()

# 5. Compile the Graph WITH the checkpointer
# This is the crucial step for persistence.
app = builder.compile(checkpointer=memory)

print("Persistent Graph compiled successfully!")

Persistent Graph compiled successfully!


## 2. Using Thread IDs to Resume Conversations

Now let's interact with our compiled graph. We will pass a `configurable` dict containing a `thread_id` to tell LangGraph which conversation we are resuming.

In [3]:
# Define a configuration with a unique thread ID
config_user_1 = {"configurable": {"thread_id": "user_1_thread_abc123"}}

print("--- Interaction 1 (User 1) ---")
input_message_1 = HumanMessage(content="Hi, my name is Alice and I love AI Engineering.")
# Invoke the graph. The checkpointer automatically saves the state afterward.
output_1 = app.invoke({"messages": [input_message_1]}, config=config_user_1)
print(f"Agent: {output_1['messages'][-1].content}\n")

print("--- Interaction 2 (User 1) ---")
input_message_2 = HumanMessage(content="What is my name and what do I love?")
# We use the SAME config. LangGraph loads the history automatically.
output_2 = app.invoke({"messages": [input_message_2]}, config=config_user_1)
print(f"Agent: {output_2['messages'][-1].content}\n")


# Let's show that another user has a separate state
config_user_2 = {"configurable": {"thread_id": "user_2_thread_xyz789"}}

print("--- Interaction 3 (User 2) ---")
input_message_3 = HumanMessage(content="What is my name?")
output_3 = app.invoke({"messages": [input_message_3]}, config=config_user_2)
print(f"Agent: {output_3['messages'][-1].content}\n")

--- Interaction 1 (User 1) ---


[Mock] LLM API Call Failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy-key. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}
Agent: I am a mock response because the API key is invalid.

--- Interaction 2 (User 1) ---
[Mock] LLM API Call Failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy-key. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}
Agent: I am a mock response because the API key is invalid.

--- Interaction 3 (User 2) ---
[Mock] LLM API Call Failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy-key. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': N

## 3. Practical Lab / Homework

Your task is to build a "Counter Agent". This agent doesn't use an LLM. Instead, it simply counts how many times the user has sent a message in a specific thread.

**Requirements:**
1.  Define a new `CounterState` typed dictionary. It should have a `count` field (integer) and a `messages` field (list of strings).
2.  Define a node function `increment_counter` that takes the `CounterState`. It should:
    - Increment the `count` by 1.
    - Append the incoming message to the `messages` list.
    - Return the updated state `{"count": new_count, "messages": [new_message]}` (Note: depending on how you define your state, you might need to handle the list appendage manually or use `operator.add` in `Annotated`).
3.  Build a `StateGraph` using this state and node.
4.  Compile the graph with a `MemorySaver` checkpointer.
5.  Invoke the graph 3 times with the **same** `thread_id` (e.g., "counter_thread_1"). Send messages like "Hello", "Are you there?", "Counting...".
6.  Print the final state to verify the `count` is 3 and all 3 messages are stored.

In [4]:
import operator
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# 1. Define State
# We use operator.add to append items to the list automatically during state updates.
class CounterState(TypedDict):
    count: int
    messages: Annotated[list[str], operator.add]

# 2. Define Node Function
def increment_counter(state: CounterState):
    """Increments the count and returns the new state."""
    # Handle the initial case where count might not exist yet
    current_count = state.get("count", 0)
    
    # We don't actually need to process the incoming message string here,
    # because it's already in the state. We just return the incremented count.
    # The 'messages' list is handled by the user passing it in during invoke.
    return {"count": current_count + 1}

# 3 & 4. Build and Compile Graph
counter_builder = StateGraph(CounterState)
counter_builder.add_node("counter_node", increment_counter)
counter_builder.add_edge(START, "counter_node")
counter_builder.add_edge("counter_node", END)

counter_memory = MemorySaver()
counter_app = counter_builder.compile(checkpointer=counter_memory)

# 5. Invoke the graph 3 times
thread_config = {"configurable": {"thread_id": "counter_thread_1"}}

print("--- Invocation 1 ---")
# Notice we must provide the initial count if the node expects it, or handle it in the node.
# We handle it in the node with .get("count", 0).
# For messages, because we use operator.add, we pass a list containing the new message.
res1 = counter_app.invoke({"messages": ["Hello"]}, config=thread_config)
print(f"State after 1: count={res1['count']}, messages={res1['messages']}")

print("--- Invocation 2 ---")
res2 = counter_app.invoke({"messages": ["Are you there?"]}, config=thread_config)
print(f"State after 2: count={res2['count']}, messages={res2['messages']}")

print("--- Invocation 3 ---")
res3 = counter_app.invoke({"messages": ["Counting..."]}, config=thread_config)
print(f"State after 3: count={res3['count']}, messages={res3['messages']}")


--- Invocation 1 ---
State after 1: count=1, messages=['Hello']
--- Invocation 2 ---
State after 2: count=2, messages=['Hello', 'Are you there?']
--- Invocation 3 ---
State after 3: count=3, messages=['Hello', 'Are you there?', 'Counting...']
